<a href="https://colab.research.google.com/github/farrelrassya/LLM-Design-Patterns-Building-Robust-and-Efficient-AI-Systems/blob/main/05.Data_Versioning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 5 — Data Versioning

*LLM Design Patterns*

Bayangkan minggu lalu model kamu menjawab pertanyaan onkologi dengan baik. Minggu ini tim menambahkan 10.000 paper baru ke dataset, model dilatih ulang, dan jawabannya memburuk. Pertanyaan pertama yang muncul: data persis apa yang melatih model minggu lalu, dan bisakah kita kembali ke sana? Kalau yang kita punya hanya file bernama `dataset_final_v2_FIXED.json`, tidak ada yang bisa dibuktikan.

Data versioning menjawab tiga kebutuhan. **Identitas**: setiap keadaan dataset punya ID yang bisa dihitung ulang dari isinya. **Riwayat**: kita tahu versi mana berubah menjadi versi mana. **Keterlacakan**: dari sebuah model kita bisa menunjuk versi data yang melatihnya, dan sebaliknya.

Notebook ini membangun ketiganya bertahap: hash sebagai identitas, delta untuk menyimpan perubahan, DVC sebagai perkakas sungguhan, lalu versi data yang ditanam ke model yang benar-benar dilatih. Beberapa listing buku tidak bisa berjalan apa adanya. Masing-masing ditaruh di kotak pendek, lalu diikuti versi yang bekerja.

**Catatan lingkungan.** Hampir seluruh bab ini pekerjaan CPU dan disk: hashing, diff teks, DVC, dan manifest file. GPU hanya dipakai di bagian 4, saat model kecil dilatih dan versi datanya ditanam ke checkpoint. Pemeriksaan GPU tetap dijalankan paling awal supaya runtime yang salah ketahuan sebelum ada instalasi atau unduhan. Output sel sengaja dikosongkan, jadi semua angka yang kamu lihat berasal dari run-mu sendiri. Penyusun memvalidasi notebook ini secara lokal di CPU dengan profil kecil; run di CUDA/Colab belum dilakukan penyusun.

## 0. Setup

Semua pengaturan dikumpulkan di sel berikut. Dua yang perlu kamu tahu: `DATA_SOURCE` (default `"c4"`, mengunduh 350 dokumen pertama dari satu shard C4 pada commit yang dipatok; ganti ke `"synthetic"` kalau tidak ada internet) dan `ALLOW_CPU_REDUCED` (default `False`; set `True` hanya kalau kamu sengaja ingin menjalankan training bagian 4 di CPU dengan profil yang jauh lebih kecil). Semua file kerja, termasuk cache dan konfigurasi DVC, ditaruh di bawah `WORK_DIR`.

In [1]:
import os
import sys
import re
import json
import time
import gzip
import shutil
import hashlib
import difflib
import subprocess
import importlib.util
import urllib.request
from pathlib import Path

import numpy as np

DATA_SOURCE = "c4"          # "c4" (unduh) atau "synthetic" (offline, teks buatan)
ALLOW_CPU_REDUCED = False   # True = sengaja izinkan training bagian 4 di CPU, profil kecil
N_DOCS_V1 = 300             # dokumen di versi 1 dataset
N_DOCS_EXTRA = 50           # dokumen baru untuk versi 2
C4_REVISION = "1588ec454efa1a09f29cd18ddd04fe05fc8653a2"   # commit repo allenai/c4 di Hugging Face
C4_FILE = "en/c4-train.00000-of-01024.json.gz"
NETWORK_TIMEOUT_S = 180     # batas total waktu unduh
DVC_VERSION = "3.67.1"
SEED = 0

IN_COLAB = "google.colab" in sys.modules
WORK_DIR = Path(os.environ.get("LDP05_WORK_DIR", "/content/ldp05" if IN_COLAB else "ldp05_work")).resolve()
WORK_DIR.mkdir(parents=True, exist_ok=True)

# konfigurasi global/sistem dan cache situs DVC diarahkan ke WORK_DIR; telemetri dan Studio DVC dimatikan
for var, sub in [("DVC_GLOBAL_CONFIG_DIR", "env/dvc-global"),
                 ("DVC_SYSTEM_CONFIG_DIR", "env/dvc-system"),
                 ("DVC_SITE_CACHE_DIR", "env/dvc-site-cache")]:
    (WORK_DIR / sub).mkdir(parents=True, exist_ok=True)
    os.environ[var] = str(WORK_DIR / sub)
os.environ["DVC_NO_ANALYTICS"] = "1"
os.environ["DVC_STUDIO_OFFLINE"] = "1"
print("work dir:", WORK_DIR, "| Colab:", IN_COLAB)

work dir: /content/ldp05 | Colab: True


Pemeriksaan GPU di bawah tidak berhenti di `torch.cuda.is_available()`. Ia benar-benar mengalokasikan dua matriks 1024×1024 di GPU, mengalikannya, menunggu GPU selesai (`synchronize`), lalu memastikan hasilnya berhingga. Dari *compute capability* kartu, sel ini juga memilih presisi: GPU generasi Ampere ke atas (capability ≥ 8, misalnya A100 atau L4) memakai bfloat16, sedangkan T4 (7.5) tidak punya dukungan bfloat16 bawaan sehingga memakai float16 dengan *gradient scaling*. Kalau CUDA tidak ada, sel berhenti dengan instruksi, kecuali kamu sendiri mengaktifkan `ALLOW_CPU_REDUCED`.

In [2]:
import torch


def cuda_functional_gate():
    if not torch.cuda.is_available():
        return None
    dev = torch.device("cuda")
    a = torch.randn(1024, 1024, device=dev)
    b = torch.randn(1024, 1024, device=dev)
    c = a @ b
    torch.cuda.synchronize()
    if not torch.isfinite(c).all().item():
        raise RuntimeError("matmul uji di GPU menghasilkan nilai tak berhingga; ganti runtime sebelum lanjut")
    free_b, total_b = torch.cuda.mem_get_info()
    major, minor = torch.cuda.get_device_capability(dev)
    del a, b, c
    torch.cuda.empty_cache()
    return {"name": torch.cuda.get_device_name(dev), "major": major, "minor": minor,
            "free_gb": free_b / 2**30, "total_gb": total_b / 2**30}


gpu = cuda_functional_gate()
if gpu is not None:
    DEVICE = torch.device("cuda")
    AMP_DTYPE = torch.bfloat16 if gpu["major"] >= 8 else torch.float16
    PROFILE = "gpu"
    print(f"GPU OK: {gpu['name']} (capability {gpu['major']}.{gpu['minor']}), "
          f"memori bebas {gpu['free_gb']:.1f} dari {gpu['total_gb']:.1f} GB")
elif ALLOW_CPU_REDUCED:
    DEVICE = torch.device("cpu")
    AMP_DTYPE = torch.float32
    PROFILE = "cpu_reduced"
    print("CUDA tidak tersedia. ALLOW_CPU_REDUCED=True, jadi training bagian 4 memakai profil CPU kecil.")
else:
    raise RuntimeError("CUDA tidak tersedia. Di Colab: Runtime -> Change runtime type -> GPU (mis. T4), "
                       "lalu jalankan ulang dari atas. Untuk sengaja memakai CPU, set ALLOW_CPU_REDUCED = True.")
print("torch", torch.__version__, "| device training:", DEVICE, "| presisi autocast:", AMP_DTYPE, "| profil:", PROFILE)

GPU OK: Tesla T4 (capability 7.5), memori bebas 14.4 dari 14.6 GB
torch 2.11.0+cu130 | device training: cuda | presisi autocast: torch.float16 | profil: gpu


DVC tidak terpasang bawaan di Colab. Kita pasang versi yang dipatok, lalu memanggilnya lewat `python -m dvc` supaya pasti memakai interpreter yang sama dengan notebook. DVC dipanggil sebagai proses terpisah, jadi modulnya tidak pernah diimpor ke kernel notebook ini.

In [3]:
if importlib.util.find_spec("dvc") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"dvc=={DVC_VERSION}"], check=True)
DVC = [sys.executable, "-m", "dvc"]
print("dvc", subprocess.run(DVC + ["--version"], check=True, capture_output=True, text=True).stdout.strip())

dvc 3.67.1


## 1. Hash sebagai identitas dataset · CPU

Cara paling sederhana memberi dataset identitas: ubah isinya menjadi teks yang selalu sama untuk isi yang sama, lalu hitung **hash** SHA-256 dari teks itu. SHA-256 memetakan input sebesar apa pun ke 64 karakter heksadesimal. Input yang sama selalu menghasilkan hash yang sama, sedangkan perubahan satu karakter saja menghasilkan hash yang tampak acak dan berbeda total. Buku membungkus ide ini dalam kelas `DatasetVersion`: data, metadata, timestamp, dan hash.

> **Listing buku: `DatasetVersion` dengan komentar gaya C**
> ```python
> self.timestamp = datetime.now().isoformat()
> //creation timestamp for each version
> self.version_hash = self._generate_hash()
> ```
> Python tidak mengenal komentar `//` (itu operator pembagian bulat), jadi seluruh sel gagal di-parse dengan `SyntaxError: invalid syntax` sebelum satu baris pun berjalan. Versi kerja di bawah memakai `#`, menggabungkan listing `save`/`load` lanjutan dari buku, dan menambahkan satu pemeriksaan integritas di `load`.

In [4]:
from datetime import datetime
import hashlib
import json


class DatasetVersion:
    def __init__(self, data, metadata=None):
        self.data = data
        self.metadata = metadata or {}
        self.timestamp = datetime.now().isoformat()  # waktu pembuatan versi; tidak ikut di-hash
        self.version_hash = self._generate_hash()

    def _generate_hash(self):
        data_str = json.dumps(self.data, sort_keys=True).encode()
        return hashlib.sha256(data_str).hexdigest()

    def save(self, filename):
        with open(filename, "w") as f:
            json.dump({
                "data": self.data,
                "metadata": self.metadata,
                "timestamp": self.timestamp,
                "version_hash": self.version_hash,
            }, f, indent=2)

    @classmethod
    def load(cls, filename):
        with open(filename) as f:
            payload = json.load(f)
        instance = cls(payload["data"], payload["metadata"])
        if instance.version_hash != payload["version_hash"]:  # tambahan: buku langsung menimpa hash ini
            raise ValueError("isi file tidak cocok dengan version_hash yang tersimpan")
        instance.timestamp = payload["timestamp"]
        return instance

Tiga versi kecil cukup untuk melihat sifat-sifatnya. A dan B berisi data yang sama, tetapi urutan key di setiap record B dibalik. C sama dengan A kecuali satu tanda seru. Setelah itu A disimpan, dimuat ulang, lalu salinannya diedit tangan untuk melihat apakah `load` menolaknya.

In [5]:
a = DatasetVersion({"docs": [{"id": 1, "text": "alpha"}, {"id": 2, "text": "beta"}]}, metadata={"source": "demo"})
b = DatasetVersion({"docs": [{"text": "alpha", "id": 1}, {"text": "beta", "id": 2}]})
c = DatasetVersion({"docs": [{"id": 1, "text": "alpha"}, {"id": 2, "text": "beta!"}]})
print("A:", a.version_hash[:16])
print("B:", b.version_hash[:16], "| sama dengan A:", a.version_hash == b.version_hash)
print("C:", c.version_hash[:16], "| sama dengan A:", a.version_hash == c.version_hash)
assert a.version_hash == b.version_hash != c.version_hash

path_a = WORK_DIR / "version_a.json"
a.save(path_a)
assert DatasetVersion.load(path_a).version_hash == a.version_hash
print("save -> load: hash cocok")

tampered = json.loads(path_a.read_text())
tampered["data"]["docs"][0]["text"] = "alpha (diedit tangan)"
path_bad = WORK_DIR / "version_a_tampered.json"
path_bad.write_text(json.dumps(tampered))
try:
    DatasetVersion.load(path_bad)
    raise AssertionError("file yang diedit seharusnya ditolak")
except ValueError as err:
    print("file yang diedit ditolak:", err)

A: 48c060e3153bb95e
B: 48c060e3153bb95e | sama dengan A: True
C: e17e291dd0a20cbe | sama dengan A: False
save -> load: hash cocok
file yang diedit ditolak: isi file tidak cocok dengan version_hash yang tersimpan


A dan B mendapat hash yang sama walaupun urutan key-nya berbeda, karena `sort_keys=True` menyeragamkan urutan sebelum data di-hash. Tanpa itu, dua dict yang isinya identik bisa menjadi JSON yang berbeda dan dianggap dua versi. C berbeda total walaupun yang berubah hanya satu karakter, jadi perubahan sekecil apa pun tidak bisa lolos diam-diam. Timestamp disimpan sebagai metadata dan tidak ikut di-hash, sehingga data yang sama yang dibuat ulang besok tetap dikenali sebagai versi yang sama.

Pemeriksaan di `load` adalah satu-satunya tambahan kita. Listing buku menghitung ulang hash lalu langsung menimpanya dengan hash dari file, sehingga file yang diedit tangan akan dimuat tanpa ketahuan.

Kelemahan skema ini ada di biaya: setiap versi menyimpan salinan penuh. Kalau dataset 1 GB berubah 0,1% per revisi, sepuluh revisi memakan 10 GB walaupun yang benar-benar baru hanya sekitar 10 MB. Ide berikutnya: simpan bedanya saja.

## 2. Delta: menyimpan bedanya saja · CPU

Versioning berbasis **delta** menyimpan satu versi penuh sebagai *base*, lalu versi berikutnya hanya sebagai daftar perubahan baris terhadap base. Pendekatan ini cocok untuk data tekstual yang berubah sedikit demi sedikit, profil yang umum untuk dataset LLM. Buku juga menyebut batasnya: perombakan struktur besar (ganti skema, pecah file) dan file biner menghapus keunggulan delta. Buku memakai modul `difflib` bawaan Python, dan ada tiga fungsinya yang perlu dikenal:

- `difflib.ndiff(lama, baru)` menghasilkan diff yang memuat *semua* baris kedua versi. Setiap baris diberi awalan dua karakter: `"  "` untuk baris yang sama, `"- "` untuk yang dihapus, `"+ "` untuk yang ditambah, dan `"? "` untuk baris petunjuk posisi perubahan.
- `difflib.restore(diff, which)` membaca diff format `ndiff` dan mengembalikan versi 1 (lama) atau 2 (baru).
- `difflib.unified_diff(lama, baru)` menghasilkan format ringkas ala `git diff`: hanya baris yang berubah plus tiga baris konteks. Format ini tidak punya fungsi penerap di `difflib`.

> **Listing buku: `DeltaDatasetVersion.load` dengan `restore` tiga argumen**
> ```python
> diff = list(difflib.unified_diff(base_data, current_data, lineterm=''))
> ...
> patched_data = difflib.restore(base_data, data['delta'].splitlines(), 1)
> ```
> `restore` hanya menerima dua argumen, jadi baris ini gagal dengan `TypeError: restore() takes 2 positional arguments but 3 were given`. Membuang argumen `base_data` belum cukup: delta yang disimpan berformat `unified_diff`, dan `restore` diam-diam mengembalikan list kosong untuk format itu, sehingga error baru muncul belakangan di `json.loads` sebagai `JSONDecodeError`. Versi kerja di bawah membuat delta dengan `ndiff`, pasangan sah `restore`.

In [6]:
def json_lines(obj):
    # indent=0 memecah JSON menjadi satu baris per elemen, sehingga diff bekerja per record
    return json.dumps(obj, sort_keys=True, indent=0).splitlines()


class DeltaDatasetVersion(DatasetVersion):
    def __init__(self, data, base_version=None, metadata=None):
        super().__init__(data, metadata)
        self.base_version = base_version
        self.delta = self._compute_delta() if base_version else None

    def _compute_delta(self):
        diff = difflib.ndiff(json_lines(self.base_version.data), json_lines(self.data))
        return "\n".join(line.rstrip("\n") for line in diff)  # baris petunjuk "? " membawa newline sendiri

    def save(self, filename):
        with open(filename, "w") as f:
            json.dump({
                "metadata": self.metadata,
                "timestamp": self.timestamp,
                "version_hash": self.version_hash,
                "base_version_hash": self.base_version.version_hash if self.base_version else None,
                "delta": self.delta,
            }, f, indent=2)

    @classmethod
    def load(cls, filename, base_version):
        with open(filename) as f:
            payload = json.load(f)
        if payload["base_version_hash"] != base_version.version_hash:
            raise ValueError("base version tidak cocok dengan delta ini")
        restored = "\n".join(difflib.restore(payload["delta"].splitlines(), 2))
        instance = cls(json.loads(restored), base_version, payload["metadata"])
        if instance.version_hash != payload["version_hash"]:
            raise ValueError("hash hasil rekonstruksi tidak cocok: delta rusak")
        instance.timestamp = payload["timestamp"]
        return instance

Contoh konkretnya satu siklus kurasi kecil: dari tiga record, record 2 dihapus, record 3 direvisi, serta record 4 dan 5 ditambahkan. Kita simpan versi baru sebagai delta, memulihkannya dari base, lalu memeriksa bahwa isi, hash, jumlah record, dan skema `{id, text}` tetap konsisten.

In [7]:
def check_schema(data, required=("id", "text")):
    bad = [r for r in data["docs"] if set(r) != set(required) or not isinstance(r["text"], str)]
    if bad:
        raise ValueError(f"{len(bad)} record tidak sesuai skema {required}")


base = DatasetVersion({"docs": [{"id": 1, "text": "alpha"}, {"id": 2, "text": "beta"}, {"id": 3, "text": "gamma"}]})
new_docs = [{"id": 1, "text": "alpha"}, {"id": 3, "text": "gamma (revisi)"},
            {"id": 4, "text": "delta"}, {"id": 5, "text": "epsilon"}]
v2 = DeltaDatasetVersion({"docs": new_docs}, base_version=base)
check_schema(v2.data)
path_v2 = WORK_DIR / "v2_delta.json"
v2.save(path_v2)

restored = DeltaDatasetVersion.load(path_v2, base)
check_schema(restored.data)
assert restored.data == v2.data and restored.version_hash == v2.version_hash
print("delta ndiff yang disimpan:")
print(v2.delta)
print("\nhasil rekonstruksi:", restored.data["docs"])
print("jumlah record base -> v2:", len(base.data["docs"]), "->", len(restored.data["docs"]))

delta ndiff yang disimpan:
  {
  "docs": [
  {
  "id": 1,
  "text": "alpha"
  },
  {
- "id": 2,
?       ^
+ "id": 3,
?       ^
- "text": "beta"
+ "text": "gamma (revisi)"
  },
  {
- "id": 3,
?       ^
+ "id": 4,
?       ^
- "text": "gamma"
+ "text": "delta"
+ },
+ {
+ "id": 5,
+ "text": "epsilon"
  }
  ]
  }

hasil rekonstruksi: [{'id': 1, 'text': 'alpha'}, {'id': 3, 'text': 'gamma (revisi)'}, {'id': 4, 'text': 'delta'}, {'id': 5, 'text': 'epsilon'}]
jumlah record base -> v2: 3 -> 4


Baris berawalan `"  "` di atas adalah baris yang tidak berubah, dan semuanya ikut tersimpan di delta. Simpan pengamatan itu, karena sebentar lagi kita ukur akibatnya. Perhatikan juga bahwa diff bekerja per baris, bukan per record: ia bisa melaporkan `"id": 2` berubah menjadi `"id": 3` alih-alih "record 2 dihapus". Rekonstruksinya tetap benar, karena yang dijamin diff hanyalah urutan baris, dan hitungan record 3 → 4 cocok dengan satu penghapusan dan dua penambahan.

Kelas versi kerja punya dua pengaman yang tidak ada di buku. Sebelum menerapkan delta, ia mencocokkan hash base; menerapkan delta ke base yang salah adalah cara lain merusak data tanpa error. Sesudah rekonstruksi, ia mencocokkan hash hasil dengan hash yang tersimpan. Sel berikut menguji keduanya, lalu memperlihatkan kegagalan sunyi `restore` pada format `unified_diff`.

In [8]:
other_base = DatasetVersion({"docs": [{"id": 9, "text": "lain"}]})
broken = json.loads(path_v2.read_text())
broken["delta"] = broken["delta"].replace("gamma (revisi)", "gamma (rusak)")
path_broken = WORK_DIR / "v2_delta_broken.json"
path_broken.write_text(json.dumps(broken))

for label, path, base_used in [("base salah", path_v2, other_base), ("delta rusak", path_broken, base)]:
    try:
        DeltaDatasetVersion.load(path, base_used)
        raise AssertionError(f"{label}: seharusnya ditolak")
    except ValueError as err:
        print(f"{label:<11} -> ditolak: {err}")

unified = list(difflib.unified_diff(json_lines(base.data), json_lines(v2.data), lineterm=""))
silent = list(difflib.restore(unified, 2))
print("\nrestore() pada unified_diff mengembalikan:", silent)
assert silent == []

base salah  -> ditolak: base version tidak cocok dengan delta ini
delta rusak -> ditolak: hash hasil rekonstruksi tidak cocok: delta rusak

restore() pada unified_diff mengembalikan: []


### Under the hood: apakah delta benar-benar hemat?

Kalau `ndiff` menyimpan semua baris, apakah delta masih lebih kecil dari snapshot? Eksperimen ini membuat 5.000 record sintetis (sintetis supaya jumlah perubahan bisa dikendalikan persis), mengubah 0,1% sampai 50% di antaranya, lalu membandingkan ukuran tiga jenis delta dengan ukuran snapshot penuh. Jenis ketiga adalah tambahan kita: **patch dari opcode** `difflib.SequenceMatcher`. `SequenceMatcher` sudah menghitung blok mana yang sama dan mana yang berubah; kita cukup menyimpan blok yang berubah beserta posisinya, lalu menulis fungsi penerapnya sendiri dalam beberapa baris.

In [9]:
def make_patch(old, new):
    # simpan hanya blok yang berubah: ganti old[i1:i2] dengan baris-baris baru
    sm = difflib.SequenceMatcher(None, old, new)
    return [(i1, i2, new[j1:j2]) for tag, i1, i2, j1, j2 in sm.get_opcodes() if tag != "equal"]


def apply_patch(old, patch):
    out, pos = [], 0
    for i1, i2, new_lines in patch:
        out.extend(old[pos:i1])
        out.extend(new_lines)
        pos = i2
    out.extend(old[pos:])
    return out


assert apply_patch(json_lines(base.data), make_patch(json_lines(base.data), json_lines(v2.data))) == json_lines(v2.data)

In [10]:
rng = np.random.default_rng(42)
N_RECORDS = 5000
records = [{"id": i, "text": f"dokumen nomor {i} " + "lorem ipsum dolor sit amet " * int(rng.integers(2, 6))}
           for i in range(N_RECORDS)]
base_lines = json_lines({"docs": records})
snapshot_bytes = len("\n".join(base_lines).encode())
print(f"snapshot penuh: {snapshot_bytes / 2**20:.2f} MB, {N_RECORDS} record, {len(base_lines)} baris JSON\n")
print(f"{'record diubah':>16} | {'ndiff':>7} | {'unified':>8} | {'patch':>7}   (ukuran delta / snapshot)")
for frac in (0.001, 0.01, 0.05, 0.20, 0.50):
    k = max(1, int(N_RECORDS * frac))
    hit = set(rng.choice(N_RECORDS, size=k, replace=False).tolist())
    edited = [dict(r, text=r["text"] + " EDIT") if i in hit else r for i, r in enumerate(records)]
    curr_lines = json_lines({"docs": edited})
    nd = len("\n".join(difflib.ndiff(base_lines, curr_lines)).encode())
    ud = len("\n".join(difflib.unified_diff(base_lines, curr_lines, lineterm="")).encode())
    patch = make_patch(base_lines, curr_lines)
    assert apply_patch(base_lines, patch) == curr_lines
    pt = len(json.dumps(patch).encode())
    print(f"{k:>6} ({frac:>6.1%}) | {nd / snapshot_bytes:>7.1%} | {ud / snapshot_bytes:>8.2%} | {pt / snapshot_bytes:>7.2%}")

# representasi listing buku: JSON tanpa indent = seluruh dataset dalam SATU baris
one_line_old = [json.dumps({"docs": records[:100]}, sort_keys=True)]
one_line_new = [json.dumps({"docs": records[:97]}, sort_keys=True)]
nd_one = len("\n".join(difflib.ndiff(one_line_old, one_line_new)).encode())
print(f"\nJSON satu baris (100 record, 3 dihapus): delta ndiff = {nd_one / len(one_line_old[0].encode()):.2f}x ukuran data")

snapshot penuh: 0.67 MB, 5000 record, 20004 baris JSON

   record diubah |   ndiff |  unified |   patch   (ukuran delta / snapshot)
     5 (  0.1%) |  105.8% |    0.20% |   0.10%
    50 (  1.0%) |  107.6% |    2.28% |   1.10%
   250 (  5.0%) |  114.9% |   11.07% |   5.37%
  1000 ( 20.0%) |  142.8% |   43.86% |  21.69%
  2500 ( 50.0%) |  198.9% |  105.29% |  54.35%

JSON satu baris (100 record, 3 dihapus): delta ndiff = 2.97x ukuran data


Cara membaca tabel: 100% berarti delta sama besar dengan menyimpan ulang seluruh snapshot, dan 1% berarti delta seperseratus snapshot.

- **ndiff tidak pernah di bawah 100%.** Setiap baris yang tidak berubah ikut disalin dengan awalan dua karakter, jadi ukurannya minimal sebesar snapshot, lalu baris yang berubah menambah baris `- `, `+ `, dan petunjuk `?`. Setelah diperbaiki supaya bisa dipulihkan, strategi delta buku justru kehilangan alasan keberadaannya.
- **unified tumbuh kira-kira sebanding dengan porsi record yang diubah**, karena hanya baris berubah plus konteks yang disimpan. Setiap baris yang berubah muncul dua kali (sebagai `-` dan `+`), jadi format ini hemat selama perubahannya sedikit. Masalahnya, `difflib` tidak bisa menerapkannya kembali.
- **patch** lebih kecil lagi, karena tidak menyimpan baris lama maupun konteks; base sudah memilikinya. `assert` di dalam loop membuktikan bahwa setiap patch bisa diterapkan kembali menjadi versi baru yang persis sama.

Lihat juga baris dengan porsi perubahan terbesar: di sana keuntungan delta apa pun menipis, dan menyimpan snapshot baru mulai lebih masuk akal.

Baris terakhir memperlihatkan representasi yang dipakai listing buku: JSON tanpa `indent`, sehingga seluruh dataset menjadi satu baris. Perubahan apa pun membuat baris lama dan baris baru tersimpan utuh, jadi delta minimal dua kali ukuran data, ditambah baris petunjuk `?`. Granularitas baris harus sejalan dengan granularitas record.

Pelajarannya: sebelum menjanjikan delta yang hemat, pastikan pembuat delta dan penerapnya benar-benar sepasang. Sistem nyata memakai format patch yang ringkas sekaligus bisa diterapkan, misalnya delta di packfile Git, `xdelta`, atau `bsdiff`. Patch opcode di atas adalah versi mini dari ide yang sama, dan akan kita pakai lagi di bagian 7.

## 3. DVC: versioning dengan perkakas sungguhan · CPU/I/O

Kelas buatan sendiri cukup untuk memahami ide, tetapi tidak untuk tim: tidak ada remote bersama, tidak ada cache, tidak ada integrasi pipeline. **DVC** (Data Version Control) mengisi celah itu dengan tiga bagian. *Cache* menyimpan isi file dengan nama berupa hash MD5 isinya, sehingga konten yang sama hanya tersimpan sekali (*content-addressable storage*). *Pointer* adalah file kecil berakhiran `.dvc` di samping data, berisi hash dan ukuran; pointer inilah yang biasanya di-commit ke Git. *Remote* adalah penyimpanan bersama tempat `dvc push` mengirim isi cache.

Notebook ini **tidak memakai Git**. DVC dijalankan dengan `--no-scm`, dan peran Git (menyimpan riwayat pointer) digantikan folder `pointer_history/` berisi salinan pointer setiap versi. Cache, pointer, remote, dan checkout tetap mekanisme DVC yang asli. Remote-nya folder lokal di `WORK_DIR`, jadi tidak ada kredensial atau unggahan ke mana pun.

Bahan kita adalah 350 dokumen pertama dari satu shard C4, korpus teks web yang dipakai melatih T5. Revisi repositorinya dipatok ke satu commit, jadi setiap pembaca mengunduh byte yang sama. 300 dokumen pertama menjadi versi 1 dan 50 sisanya disimpan untuk versi 2. Setiap dokumen ditulis sebagai satu baris.

In [11]:
def fetch_c4_docs(n, revision, filename, timeout_s):
    url = f"https://huggingface.co/datasets/allenai/c4/resolve/{revision}/{filename}"
    deadline = time.monotonic() + timeout_s
    request = urllib.request.Request(url, headers={"User-Agent": "ldp05-notebook"})
    docs = []
    with urllib.request.urlopen(request, timeout=30) as resp, gzip.GzipFile(fileobj=resp) as gz:
        for line in gz:
            example = json.loads(line)
            docs.append({"id": example["url"], "text": example["text"]})
            if len(docs) == n:
                return docs
            if time.monotonic() > deadline:
                raise TimeoutError(f"unduhan C4 melewati {timeout_s} s; coba lagi atau set DATA_SOURCE = 'synthetic'")
    raise RuntimeError(f"shard hanya berisi {len(docs)} dokumen")


def synthetic_docs(n, seed=SEED):
    rng_docs = np.random.default_rng(seed)
    words = "data model versi hash dataset token training evaluasi korpus delta cache pointer".split()
    return [{"id": f"synthetic://{i}",
             "text": " ".join(rng_docs.choice(words, size=int(rng_docs.integers(40, 200))).tolist())}
            for i in range(n)]


def to_line(doc):
    return doc["text"].replace("\n", " ") + "\n"


n_total = N_DOCS_V1 + N_DOCS_EXTRA
if DATA_SOURCE == "c4":
    docs_all = fetch_c4_docs(n_total, C4_REVISION, C4_FILE, NETWORK_TIMEOUT_S)
elif DATA_SOURCE == "synthetic":
    docs_all = synthetic_docs(n_total)
else:
    raise ValueError("DATA_SOURCE harus 'c4' atau 'synthetic'")
docs_v1, docs_extra = docs_all[:N_DOCS_V1], docs_all[N_DOCS_V1:]
v1_bytes = "".join(to_line(d) for d in docs_v1).encode()
v1_sha = hashlib.sha256(v1_bytes).hexdigest()

REFERENCE_SHA256_C4_V1 = "71e8ba46c157337dd1295fb56a36a6bfac04dd82955f35e1b6eef669644d0a7f"
print(f"sumber: {DATA_SOURCE} | versi 1: {len(docs_v1)} dokumen, {len(v1_bytes) / 2**20:.2f} MB")
print("SHA-256 isi versi 1:", v1_sha)
if DATA_SOURCE == "c4":
    print("sama dengan hash yang dicatat saat notebook disusun:", v1_sha == REFERENCE_SHA256_C4_V1)

sumber: c4 | versi 1: 300 dokumen, 0.59 MB
SHA-256 isi versi 1: 71e8ba46c157337dd1295fb56a36a6bfac04dd82955f35e1b6eef669644d0a7f
sama dengan hash yang dicatat saat notebook disusun: True


Hash di atas adalah identitas data kita sendiri: SHA-256 seperti di bagian 1, kali ini langsung atas byte file. Kalau baris terakhir mencetak `True`, versi 1 milikmu identik byte demi byte dengan yang dipakai saat notebook ini disusun, jadi ukuran file dan hash yang muncul di bawah juga akan sama. Kalau `False`, datamu berbeda; semua sel tetap berjalan, hanya angkanya yang berubah.

Sekarang listing DVC dari buku. Ia membungkus perintah DVC dan Git dengan `subprocess.run`.

> **Listing buku: pembungkus DVC tanpa pemeriksaan kode keluar**
> ```python
> def initialize_dvc():
>     subprocess.run(["dvc", "init"])
>     print("DVC initialized in the current directory.")
> ```
> Di folder yang bukan repo Git, `dvc init` menolak berjalan dan keluar dengan kode error, tetapi fungsi ini tetap mencetak "DVC initialized" karena `subprocess.run` tanpa `check=True` tidak melempar exception dan kode keluarnya dibuang. `add_dataset_to_dvc` dan `commit_dataset_version` punya pola yang sama. Selain itu, `commit_dataset_version` menjalankan `git add .dvc`, yang hanya menambahkan folder konfigurasi `.dvc/`; pointer `language_dataset.txt.dvc` tidak ikut ter-stage (terlihat di output historis notebook sumber; notebook ini tidak menjalankan Git). Versi kerja di bawah memeriksa setiap kode keluar dan memakai `dvc init --no-scm`.

In [12]:
DVC_DIR = WORK_DIR / "dvcdemo"
REMOTE_DIR = WORK_DIR / "dvc-remote"
HISTORY = WORK_DIR / "pointer_history"  # di luar repo: DVC menganggap setiap *.dvc di dalam repo sebagai pointer aktif
DATA = DVC_DIR / "language_dataset.txt"
POINTER = DVC_DIR / "language_dataset.txt.dvc"


def run(*args):
    # kode keluar bukan nol = exception berisi pesan DVC, bukan laporan sukses palsu
    proc = subprocess.run(DVC + list(args), cwd=DVC_DIR, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"dvc {' '.join(args)} gagal (exit {proc.returncode}):\n{proc.stderr}")
    return proc.stdout


def md5_from(pointer_path):
    return re.search(r"md5: ([0-9a-f]{32})", Path(pointer_path).read_text()).group(1)


def pointer_md5():
    return md5_from(POINTER)


def file_md5(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


def snapshot_pointer(tag):
    shutil.copy(POINTER, HISTORY / f"{tag}.dvc")  # pengganti `git commit` untuk pointer

Versi 1: siapkan repo DVC dan remote lokal, tulis datanya, `dvc add`, simpan salinan pointer, lalu `dvc push`. Sel ini menghapus demo lama lebih dulu, jadi aman dijalankan ulang.

In [13]:
for folder in (DVC_DIR, REMOTE_DIR, HISTORY):
    shutil.rmtree(folder, ignore_errors=True)
DVC_DIR.mkdir(parents=True)
HISTORY.mkdir()

run("init", "--no-scm")
run("config", "core.analytics", "false")
run("config", "core.check_update", "false")
run("remote", "add", "-d", "localremote", str(REMOTE_DIR))

DATA.write_bytes(v1_bytes)
run("add", DATA.name)
snapshot_pointer("v1")
run("push")
print(POINTER.read_text())
assert pointer_md5() == file_md5(DATA)

outs:
- md5: 4e866d53d445e411def12cd07cf27ab9
  size: 615557
  hash: md5
  path: language_dataset.txt



Itulah seluruh isi pointer: `md5` adalah identitas versi 1 dan `size` adalah jumlah byte-nya. File sekitar seratus byte ini cukup untuk memulihkan data sebesar apa pun, selama cache atau remote masih menyimpan objeknya. `assert` terakhir menunjukkan bahwa MD5 versi DVC sama dengan MD5 biasa atas byte file, jadi kita bisa memverifikasinya sendiri tanpa DVC.

Versi 2 meniru skenario pembuka dalam ukuran kecil: 50 dokumen baru masuk ke dataset.

In [14]:
DATA.write_bytes(v1_bytes + "".join(to_line(d) for d in docs_extra).encode())
run("add", DATA.name)
snapshot_pointer("v2")
run("push")

md5 = {tag: md5_from(HISTORY / f"{tag}.dvc") for tag in ("v1", "v2")}
lines_v2 = DATA.read_bytes().count(b"\n")
remote_objects = [p for p in REMOTE_DIR.rglob("*") if p.is_file()]
print("md5 v1:", md5["v1"])
print("md5 v2:", md5["v2"])
print(f"baris v2: {lines_v2} (= {N_DOCS_V1} + {N_DOCS_EXTRA}) | objek di remote: {len(remote_objects)}")
assert lines_v2 == N_DOCS_V1 + N_DOCS_EXTRA and md5["v1"] != md5["v2"] and len(remote_objects) == 2

md5 v1: 4e866d53d445e411def12cd07cf27ab9
md5 v2: d5997ffc7f85aeb88eae725125389d0c
baris v2: 350 (= 300 + 50) | objek di remote: 2


Dua versi kini punya hash berbeda dan remote menyimpan dua objek, satu per versi. Anggap model yang dilatih dengan v2 ternyata memburuk. Rollback ke v1 butuh dua langkah: kembalikan pointer lama (di Git: `git checkout <commit> -- language_dataset.txt.dvc`), lalu `dvc checkout` mengambil isi yang cocok dari cache.

In [15]:
shutil.copy(HISTORY / "v1.dvc", POINTER)
run("checkout", "--force", POINTER.name)
print("md5 file di disk:", file_md5(DATA), "| cocok dengan v1:", file_md5(DATA) == md5["v1"])
print("baris setelah rollback:", DATA.read_bytes().count(b"\n"))
assert DATA.read_bytes() == v1_bytes

md5 file di disk: 4e866d53d445e411def12cd07cf27ab9 | cocok dengan v1: True
baris setelah rollback: 300


File di disk kini identik byte demi byte dengan v1, dan rollback-nya tidak menyalin data dari mana pun selain cache DVC. Riwayat versi hanya berisi pointer; datanya sendiri tinggal di cache dan remote.

> **Alur yang sama dengan Git (listing, tidak dijalankan di notebook ini)**
> ```bash
> git init && dvc init
> dvc add language_dataset.txt
> git add language_dataset.txt.dvc .gitignore   # pointer + .gitignore, bukan cuma .dvc/
> git commit -m "dataset v1"
> dvc remote add -d storage /path/ke/remote && git commit -am "remote" && dvc push
> # ubah data -> dvc add -> git commit -> dvc push
> git checkout <commit-v1> -- language_dataset.txt.dvc && dvc checkout   # rollback
> ```
> Perbedaannya dengan sel di atas hanya siapa yang menyimpan riwayat pointer: Git, bukan folder `pointer_history/`.

### Under the hood: berapa harga satu baris di cache DVC?

Cache DVC menyimpan setiap versi file sebagai objek utuh. Apa akibatnya kalau yang berubah hanya satu baris? Kita edit satu baris di v1 untuk membuat v3, lalu bandingkan pertumbuhan cache dengan ukuran delta unified untuk perubahan yang sama.

In [16]:
def dir_bytes(path):
    return sum(p.stat().st_size for p in Path(path).rglob("*") if p.is_file())


cache_dir = DVC_DIR / ".dvc" / "cache"
cache_before = dir_bytes(cache_dir)
lines = DATA.read_bytes().split(b"\n")
lines[0] = lines[0] + b" [edited]"
DATA.write_bytes(b"\n".join(lines))
run("add", DATA.name)
snapshot_pointer("v3")
cache_growth = dir_bytes(cache_dir) - cache_before

old_text = v1_bytes.decode().split("\n")
new_text = DATA.read_bytes().decode().split("\n")
unified_bytes = len("\n".join(difflib.unified_diff(old_text, new_text, lineterm="")).encode())
print(f"ukuran file v3          : {DATA.stat().st_size / 1024:9.1f} KB")
print(f"pertumbuhan cache DVC   : {cache_growth / 1024:9.1f} KB")
print(f"delta unified (1 baris) : {unified_bytes / 1024:9.1f} KB")
print(f"cache / delta           : {cache_growth / unified_bytes:9.0f}x")
assert cache_growth == DATA.stat().st_size

ukuran file v3          :     601.1 KB
pertumbuhan cache DVC   :     601.1 KB
delta unified (1 baris) :       4.2 KB
cache / delta           :       143x


`assert` terakhir memastikan pertumbuhan cache sama persis dengan ukuran satu file utuh: DVC tidak menyimpan perubahannya, melainkan versi baru seluruhnya. Delta unified untuk perubahan yang sama hanya berisi baris yang diedit beserta beberapa baris konteks di sekitarnya, dan baris `cache / delta` menunjukkan berapa kali lipat harga yang dibayar cache untuk satu baris itu.

Ini pilihan desain, bukan cacat. Dengan objek utuh, checkout tidak perlu merangkai patch, dan file biner atau data yang strukturnya dirombak tidak merugikan apa-apa; dua kasus terakhir itu justru kelemahan delta yang diakui buku. Untuk dataset yang sering berubah sedikit, praktik yang umum adalah memecah data menjadi banyak file kecil (*shard*), sehingga satu edit hanya menggandakan satu shard. Pointer kini menunjuk ke v3, dan versi inilah yang akan melatih model di bagian berikutnya.

## 4. Menanam versi data ke model · GPU

Versioning baru bermanfaat kalau terpasang di jalur training: setiap model yang keluar harus membawa identitas data yang melatihnya. Pola buku sederhana: baca info versi dari `dataset_info.json`, catat hash-nya saat training, dan sisipkan delapan karakter pertamanya ke nama file model.

> **Listing buku: blok pemakaian `train_llm`**
> ```python
> dataset_info = load_dataset_info("dataset_info.json")
> dataset = load_dataset()  # Your dataset loading function
> model = initialize_model()  # Your model initialization function
> train_llm(model, dataset, dataset_info)
> ```
> Ini sketsa, bukan program. Tidak ada listing yang membuat `dataset_info.json`, jadi baris pertama gagal dengan `FileNotFoundError`; kalaupun file itu ada, `load_dataset` dan `initialize_model` tidak pernah didefinisikan (`NameError`), dan `train_llm` buku hanya berisi komentar "Actual training code would go here". Di bawah, semuanya diisi: info versi diambil dari pointer DVC v3, dataset dibaca dari file yang dilacak DVC, dan model kecil benar-benar dilatih di GPU.

In [17]:
from dataclasses import dataclass
from typing import Any, Dict


@dataclass
class DatasetInfo:
    version_hash: str
    metadata: Dict[str, Any]


def load_dataset_info(filename) -> DatasetInfo:
    with open(filename) as f:
        data = json.load(f)
    return DatasetInfo(data["version_hash"], data["metadata"])


INFO_PATH = WORK_DIR / "dataset_info.json"
INFO_PATH.write_text(json.dumps({
    "version_hash": pointer_md5(),
    "metadata": {"source": DATA_SOURCE, "pointer": POINTER.name, "hash": "md5 (DVC)",
                 "n_lines": DATA.read_bytes().count(b"\n"), "n_bytes": DATA.stat().st_size},
}, indent=2))
dataset_info = load_dataset_info(INFO_PATH)
print(dataset_info)

DatasetInfo(version_hash='efb9529e23906c6a68ab75f7a8ad4030', metadata={'source': 'c4', 'pointer': 'language_dataset.txt.dvc', 'hash': 'md5 (DVC)', 'n_lines': 300, 'n_bytes': 615566})


Model yang kita latih sengaja kecil: transformer dua lapis yang membaca teks per byte (kosakata 256) dan menebak byte berikutnya. Ini bukan LLM yang berguna, tetapi punya semua bagian yang penting untuk pelacakan: bobot, data latih, device, dan checkpoint. Ukuran kesalahannya *cross-entropy loss* dalam nat per byte. Tebakan merata atas 256 kemungkinan memberi loss ln 256 ≈ 5,55, jadi angka di bawah itu berarti model sudah menangkap pola teks.

Profil `gpu` adalah default. Profil `cpu_reduced` hanya terpakai kalau kamu mengaktifkan `ALLOW_CPU_REDUCED`; ukurannya diperkecil supaya selesai dalam hitungan detik, dan hasilnya tidak untuk dibandingkan dengan profil GPU.

In [18]:
import contextlib
import math
import torch.nn as nn
import torch.nn.functional as F

PROFILES = {
    "gpu": dict(d_model=128, n_layers=2, n_heads=4, ctx=128, batch=64, steps=300, eval_batches=20, lr=3e-3),
    "cpu_reduced": dict(d_model=64, n_layers=2, n_heads=4, ctx=64, batch=16, steps=40, eval_batches=5, lr=3e-3),
}
CFG = PROFILES[PROFILE]


class Block(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.ln1 = nn.LayerNorm(d_model)
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.proj = nn.Linear(d_model, d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))

    def forward(self, x):
        B, T, D = x.shape
        qkv = self.qkv(self.ln1(x)).view(B, T, 3, self.n_heads, D // self.n_heads)
        q, k, v = qkv.permute(2, 0, 3, 1, 4)
        att = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.proj(att.transpose(1, 2).reshape(B, T, D))
        return x + self.mlp(self.ln2(x))


class TinyByteLM(nn.Module):
    def __init__(self, d_model, n_layers, n_heads, ctx, vocab=256):
        super().__init__()
        self.tok = nn.Embedding(vocab, d_model)
        self.pos = nn.Embedding(ctx, d_model)
        self.blocks = nn.ModuleList(Block(d_model, n_heads) for _ in range(n_layers))
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab)

    def forward(self, x):
        h = self.tok(x) + self.pos(torch.arange(x.size(1), device=x.device))
        for block in self.blocks:
            h = block(h)
        return self.head(self.norm(h))


def autocast_ctx():
    if DEVICE.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=AMP_DTYPE)
    return contextlib.nullcontext()


def sample_batch(data, generator):
    starts = torch.randint(0, data.numel() - CFG["ctx"] - 1, (CFG["batch"],), device=DEVICE, generator=generator)
    window = data[starts[:, None] + torch.arange(CFG["ctx"] + 1, device=DEVICE)].long()
    return window[:, :-1], window[:, 1:]


@torch.no_grad()
def eval_loss(model, data):
    model.eval()
    g = torch.Generator(device=DEVICE).manual_seed(SEED + 1)  # batch evaluasi yang sama setiap kali
    losses = []
    for _ in range(CFG["eval_batches"]):
        x, y = sample_batch(data, g)
        with autocast_ctx():
            logits = model(x)
        losses.append(F.cross_entropy(logits.float().reshape(-1, 256), y.reshape(-1)).item())
    model.train()
    return sum(losses) / len(losses)

Sekarang tiga fungsi yang di buku hanya berupa nama. `load_dataset` lebih dulu memastikan MD5 file di disk sama dengan versi yang tercatat di `dataset_info`; kalau tidak, training diblokir. Setelah itu byte file dipindah ke GPU dan dibagi 90% latih dan 10% validasi. `train_llm` mempertahankan signature buku, mencatat versi data, melatih model, lalu menyimpan checkpoint yang membawa hash versi data di dalamnya, bukan hanya di nama file.

In [19]:
def assert_data_matches(info, data_path):
    actual = file_md5(data_path)
    if actual != info.version_hash:
        raise ValueError(f"data di disk ({actual[:8]}) bukan versi yang tercatat ({info.version_hash[:8]})")


def load_dataset(info, data_path):
    assert_data_matches(info, data_path)  # blokir training kalau datanya bukan versi yang tercatat
    raw = torch.frombuffer(bytearray(Path(data_path).read_bytes()), dtype=torch.uint8)
    split = int(0.9 * raw.numel())
    return raw[:split].to(DEVICE), raw[split:].to(DEVICE)


def initialize_model():
    torch.manual_seed(SEED)
    return TinyByteLM(CFG["d_model"], CFG["n_layers"], CFG["n_heads"], CFG["ctx"]).to(DEVICE)


def train_llm(model, dataset, dataset_info: DatasetInfo):
    print(f"Training model with dataset version: {dataset_info.version_hash}")
    print(f"Dataset metadata: {dataset_info.metadata}")
    train_data, val_data = dataset
    param_device = next(model.parameters()).device
    assert param_device.type == DEVICE.type == train_data.device.type == val_data.device.type
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"])
    scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
    g = torch.Generator(device=DEVICE).manual_seed(SEED)
    loss_before = eval_loss(model, val_data)
    t0 = time.perf_counter()
    for step in range(CFG["steps"]):
        x, y = sample_batch(train_data, g)
        with autocast_ctx():
            logits = model(x)
        loss = F.cross_entropy(logits.float().reshape(-1, 256), y.reshape(-1))
        if not torch.isfinite(loss):
            raise FloatingPointError(f"loss tidak berhingga di step {step}")
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    train_seconds = time.perf_counter() - t0
    loss_after = eval_loss(model, val_data)
    path = WORK_DIR / f"model_trained_on_{dataset_info.version_hash[:8]}.pt"
    torch.save({
        "model_state": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "config": CFG,
        "dataset_version": dataset_info.version_hash,
        "dataset_metadata": dataset_info.metadata,
        "trained_on": torch.cuda.get_device_name() if DEVICE.type == "cuda" else "cpu",
        "loss_before": loss_before,
        "loss_after": loss_after,
    }, path)
    return path, loss_before, loss_after, train_seconds

In [20]:
dataset = load_dataset(dataset_info, DATA)
model = initialize_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"parameter: {n_params:,} | model di {next(model.parameters()).device} | data di {dataset[0].device}")
ckpt_path, loss_before, loss_after, train_seconds = train_llm(model, dataset, dataset_info)
print(f"\nloss validasi: {loss_before:.3f} -> {loss_after:.3f} nat/byte (tebakan merata = {math.log(256):.3f})")
print(f"waktu training: {train_seconds:.1f} s untuk {CFG['steps']} step, profil {PROFILE}")
print("checkpoint:", ckpt_path.name)
assert math.isfinite(loss_after) and loss_after < loss_before

parameter: 478,976 | model di cuda:0 | data di cuda:0
Training model with dataset version: efb9529e23906c6a68ab75f7a8ad4030
Dataset metadata: {'source': 'c4', 'pointer': 'language_dataset.txt.dvc', 'hash': 'md5 (DVC)', 'n_lines': 300, 'n_bytes': 615566}

loss validasi: 5.722 -> 2.184 nat/byte (tebakan merata = 5.545)
waktu training: 2.4 s untuk 300 step, profil gpu
checkpoint: model_trained_on_efb9529e.pt


Loss validasi awal mestinya dekat 5,55 karena model acak belum lebih baik dari tebakan merata. Penurunannya adalah bukti bahwa training benar-benar terjadi, bukan sekadar mencetak hash. Pertanyaan pentingnya sekarang dibalik: diberi file checkpoint saja, bisakah kita membuktikan data apa yang melatihnya? Sel berikut memuat ulang checkpoint ke device yang sama, membandingkan tiga identitas, lalu mencoba memulai training dengan info versi v2 padahal data di disk adalah v3.

In [21]:
ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=True)
arch = {k: ckpt["config"][k] for k in ("d_model", "n_layers", "n_heads", "ctx")}
reloaded = TinyByteLM(**arch).to(DEVICE)
reloaded.load_state_dict(ckpt["model_state"])
reloaded_loss = eval_loss(reloaded, dataset[1])
print("versi data di checkpoint :", ckpt["dataset_version"])
print("versi data di pointer DVC:", pointer_md5())
print("MD5 file data saat ini   :", file_md5(DATA))
print(f"loss model yang dimuat ulang: {reloaded_loss:.3f} (saat disimpan: {ckpt['loss_after']:.3f})")
assert ckpt["dataset_version"] == pointer_md5() == file_md5(DATA)
assert abs(reloaded_loss - ckpt["loss_after"]) < 1e-3

info_v2 = DatasetInfo(md5["v2"], {"catatan": "info versi lama"})
try:
    load_dataset(info_v2, DATA)
    raise AssertionError("data v3 seharusnya ditolak untuk info v2")
except ValueError as err:
    print("\ntraining diblokir:", err)

del model, reloaded, dataset
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

versi data di checkpoint : efb9529e23906c6a68ab75f7a8ad4030
versi data di pointer DVC: efb9529e23906c6a68ab75f7a8ad4030
MD5 file data saat ini   : efb9529e23906c6a68ab75f7a8ad4030
loss model yang dimuat ulang: 2.184 (saat disimpan: 2.184)

training diblokir: data di disk (efb9529e) bukan versi yang tercatat (d5997ffc)


Ketiga identitas sama, jadi pertanyaan "model ini dilatih dengan data apa?" terjawab dari checkpoint itu sendiri, dan jawabannya bisa dibuktikan dengan menghitung ulang hash file data. Nama `model_trained_on_<8 karakter>.pt` hanya jalan pintas untuk manusia. Sumber kebenarannya adalah field `dataset_version` di dalam checkpoint, karena nama file mudah diganti. Loss model yang dimuat ulang sama dengan saat disimpan, tanda bahwa checkpoint utuh.

Percobaan terakhir adalah versi mini dari pemeriksaan CI/CD: sebelum training, hitung ulang hash data dan tolak kalau tidak sama dengan versi yang disetujui. Di sistem nyata, hash yang sama juga dicatat di experiment tracker dan model registry, sehingga arah sebaliknya (versi data → semua model yang memakainya) cukup dicari di registry.

## 5. Manifest untuk korpus berbentuk direktori · CPU/I/O

Dataset LLM jarang berupa satu file; lebih sering berupa direktori berisi ribuan dokumen. Pendekatan buku adalah **manifest**: kamus `path relatif → SHA-256` untuk setiap file. Membandingkan dua manifest langsung memberi tahu file mana yang ditambah, dihapus, atau diubah, tanpa membandingkan isi file satu per satu.

Kita tulis 300 dokumen versi 1 sebagai file terpisah di `corpus_v1`, lalu membuat `corpus_v2` dengan satu siklus kurasi yang jawabannya kita tahu persis: 4 dokumen direvisi, 3 dokumen baru masuk, dan 2 dokumen dibuang.

In [22]:
CORPUS_V1 = WORK_DIR / "corpus_v1"
CORPUS_V2 = WORK_DIR / "corpus_v2"
for folder in (CORPUS_V1, CORPUS_V2):
    shutil.rmtree(folder, ignore_errors=True)
CORPUS_V1.mkdir()
for i, doc in enumerate(docs_v1):
    (CORPUS_V1 / f"doc_{i:04d}.txt").write_text(doc["text"], encoding="utf-8")
shutil.copytree(CORPUS_V1, CORPUS_V2)

planted = {
    "added": [f"doc_{i:04d}.txt" for i in (300, 301, 302)],
    "removed": ["doc_0005.txt", "doc_0006.txt"],
    "modified": [f"doc_{i:04d}.txt" for i in (10, 20, 30, 40)],
}
for name in planted["modified"]:
    with open(CORPUS_V2 / name, "a", encoding="utf-8") as f:
        f.write(" [revisi editorial]")
for name, doc in zip(planted["added"], docs_extra):
    (CORPUS_V2 / name).write_text(doc["text"], encoding="utf-8")
for name in planted["removed"]:
    (CORPUS_V2 / name).unlink()

size_v1 = sum(p.stat().st_size for p in CORPUS_V1.iterdir())
print(f"corpus_v1: {len(list(CORPUS_V1.iterdir()))} file, {size_v1 / 2**20:.2f} MB | "
      f"corpus_v2: {len(list(CORPUS_V2.iterdir()))} file")

corpus_v1: 300 file, 0.59 MB | corpus_v2: 301 file


Fungsi manifest dan pembandingnya berasal dari buku dengan dua penyesuaian kecil: file dibaca per 1 MB (listing buku memuat file utuh ke memori dengan `f.read()`, masalah untuk file berukuran GB), dan variabel loop `hash` diganti `digest` supaya tidak menimpa fungsi bawaan Python. Fungsi terakhir adalah tambahan kita: hash dari manifest itu sendiri.

In [23]:
from typing import Dict, List


def hash_file(filepath, chunk_size=2**20) -> str:
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def generate_corpus_manifest(corpus_dir) -> Dict[str, str]:
    manifest = {}
    for root, _, files in os.walk(corpus_dir):
        for file in files:
            filepath = os.path.join(root, file)
            manifest[os.path.relpath(filepath, corpus_dir)] = hash_file(filepath)
    return manifest


def compare_manifests(old_manifest: Dict[str, str], new_manifest: Dict[str, str]) -> Dict[str, List[str]]:
    changes = {"added": [], "removed": [], "modified": []}
    for file, digest in new_manifest.items():
        if file not in old_manifest:
            changes["added"].append(file)
        elif old_manifest[file] != digest:
            changes["modified"].append(file)
    for file in old_manifest:
        if file not in new_manifest:
            changes["removed"].append(file)
    return changes


def manifest_fingerprint(manifest):
    return hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()


old_manifest = generate_corpus_manifest(CORPUS_V1)
new_manifest = generate_corpus_manifest(CORPUS_V2)
changes = compare_manifests(old_manifest, new_manifest)
print("Corpus changes:")
for change_type, files in changes.items():
    print(f"  {change_type:<9} ({len(files)}): {sorted(files)}")
print("\nsidik jari korpus v1:", manifest_fingerprint(old_manifest)[:16])
print("sidik jari korpus v2:", manifest_fingerprint(new_manifest)[:16])
assert {k: sorted(v) for k, v in changes.items()} == {k: sorted(v) for k, v in planted.items()}

Corpus changes:
  added     (3): ['doc_0300.txt', 'doc_0301.txt', 'doc_0302.txt']
  removed   (2): ['doc_0005.txt', 'doc_0006.txt']
  modified  (4): ['doc_0010.txt', 'doc_0020.txt', 'doc_0030.txt', 'doc_0040.txt']

sidik jari korpus v1: d5348cd1e20b8cef
sidik jari korpus v2: 821038805f52a7ce


Sembilan perubahan yang kita tanam terdeteksi tepat per kategori, dan `assert` memastikan tidak ada file lain yang ikut terlapor. Sidik jari korpus memampatkan 300 hash menjadi satu ID untuk seluruh direktori. Mirip cara DVC memberi satu hash untuk direktori yang di-`dvc add` (lewat file daftar `.dir` berisi path dan MD5 setiap file). Dua korpus dengan sidik jari sama praktis dijamin berisi file yang sama dengan isi yang sama.

Semua skema di atas bergantung pada satu operasi: menghitung hash seluruh data. Buku mengingatkan bahwa di skala LLM, checksum bisa menjadi bottleneck. Seberapa mahal sebenarnya?

### Under the hood: biaya hashing di skala korpus

Kita ukur throughput empat algoritma pada 64 MB data acak di satu core CPU, lalu menerjemahkannya ke waktu satu pass atas 750 GB, kira-kira ukuran C4 bahasa Inggris menurut makalah T5. Terakhir, waktu membuat manifest `corpus_v1` menunjukkan throughput efektif ketika data terpecah menjadi banyak file kecil.

In [24]:
payload = memoryview(os.urandom(64 * 2**20))
print(f"{'algoritma':<9} | {'MB/s':>6} | jam untuk 750 GB (1 core)")
for name in ("md5", "sha1", "sha256", "blake2b"):
    h = hashlib.new(name)
    t0 = time.perf_counter()
    for off in range(0, len(payload), 2**20):
        h.update(payload[off:off + 2**20])
    mbps = 64 / (time.perf_counter() - t0)
    print(f"{name:<9} | {mbps:>6.0f} | {750 * 1024 / mbps / 3600:5.2f}")

t0 = time.perf_counter()
generate_corpus_manifest(CORPUS_V1)
dt = time.perf_counter() - t0
print(f"\nmanifest corpus_v1 ({size_v1 / 2**20:.2f} MB, {N_DOCS_V1} file): {dt * 1000:.0f} ms "
      f"= {size_v1 / 2**20 / dt:.0f} MB/s efektif")

algoritma |   MB/s | jam untuk 750 GB (1 core)
md5       |    589 |  0.36
sha1      |    841 |  0.25
sha256    |    374 |  0.57
blake2b   |    524 |  0.41

manifest corpus_v1 (0.59 MB, 300 file): 8 ms = 71 MB/s efektif


Kolom MB/s adalah throughput satu core, dan kolom jam adalah 750 × 1024 MB dibagi throughput itu. Urutan algoritma bergantung pada CPU. Prosesor dengan instruksi khusus SHA (SHA-NI di x86, ekstensi kripto di ARM) bisa membuat SHA-1 dan SHA-256 lebih cepat dari MD5, sedangkan tanpa instruksi itu MD5 biasanya menang. Jadi anggapan "MD5 itu hash yang murah" perlu dicek di mesin target, bukan dipercaya begitu saja. Angka manifest biasanya lebih rendah dari throughput mentah karena setiap file kecil menambah biaya buka, baca, dan tutup.

Yang tidak bergantung CPU: satu pass atas ratusan GB tetap memakan waktu yang berarti per core, dan itu untuk setiap kali seluruh data di-hash ulang. Karena itu perkakas seperti DVC tidak menghitung ulang semua hash setiap kali `dvc status`; mereka menyimpan hash bersama metadata file (ukuran, waktu modifikasi, inode) dan hanya menghitung ulang file yang metadatanya berubah. Manifest juga mudah diparalelkan, karena setiap file bisa di-hash secara independen.

## 6. Varian dataset untuk eksperimen · CPU

Eksperimen jarang memakai satu dataset saja. Ada versi dasar, versi yang diperbesar, versi multibahasa, versi yang difilter ulang. Buku menyimpannya di registry JSON: setiap varian hanya mencatat induknya (`base`) dan perbedaannya (`changes`), lalu konfigurasi lengkap dihitung dengan menelusuri induk secara rekursif. Ini ide delta lagi, kali ini untuk konfigurasi, bukan isi data.

> **Listing buku: `get_variant` membuat set, bukan menggabungkan dict**
> ```python
> base_data = self.get_variant(variant["base"]) \
>     if variant["base"] else {}
> return {base_data, variant["changes"]}
> ```
> `{a, b}` adalah literal *set* berisi dua dict, dan dict tidak bisa menjadi anggota set, sehingga pemanggilan pertama gagal dengan `TypeError: unhashable type: 'dict'` (redaksi pesannya sedikit berbeda antarversi Python). Yang dimaksud prosa buku adalah menerapkan perubahan dari induk secara rekursif, yaitu `{**base_data, **variant["changes"]}`. Versi kerja di bawah memakai merge itu dan menambahkan deteksi siklus.

In [25]:
from typing import Any, Dict


class DatasetVariantManager:
    def __init__(self, base_path):
        self.base_path = base_path
        self.variants: Dict[str, Dict[str, Any]] = {}
        self._load_variants()

    def _load_variants(self):
        registry = os.path.join(self.base_path, "variants.json")
        if os.path.exists(registry):
            with open(registry) as f:
                self.variants = json.load(f)

    def save_variants(self):
        with open(os.path.join(self.base_path, "variants.json"), "w") as f:
            json.dump(self.variants, f, indent=2)

    def create_variant(self, name: str, base_variant, changes: Dict[str, Any]):
        if name in self.variants:
            raise ValueError(f"Variant {name} already exists")
        self.variants[name] = {"base": base_variant, "changes": changes}
        self.save_variants()

    def get_variant(self, name: str, _chain=()) -> Dict[str, Any]:
        if name not in self.variants:
            raise ValueError(f"Variant {name} does not exist")
        if name in _chain:
            raise ValueError("siklus varian: " + " -> ".join(_chain + (name,)))
        variant = self.variants[name]
        base_data = self.get_variant(variant["base"], _chain + (name,)) if variant["base"] else {}
        return {**base_data, **variant["changes"]}  # merge: perubahan anak menimpa induk

In [26]:
VARIANT_DIR = WORK_DIR / "variants_dir"
shutil.rmtree(VARIANT_DIR, ignore_errors=True)
VARIANT_DIR.mkdir()
manager = DatasetVariantManager(VARIANT_DIR)
manager.create_variant("base", None, {"size": 1000000, "language": "en"})
manager.create_variant("large", "base", {"size": 5000000})
manager.create_variant("multilingual", "large", {"language": ["en", "es", "fr"]})
for name in ("base", "large", "multilingual"):
    print(f"{name:<13} ->", manager.get_variant(name))
print("\nregistry di disk (variants.json):")
print((VARIANT_DIR / "variants.json").read_text())
assert manager.get_variant("multilingual") == {"size": 5000000, "language": ["en", "es", "fr"]}
assert DatasetVariantManager(VARIANT_DIR).variants == manager.variants

base          -> {'size': 1000000, 'language': 'en'}
large         -> {'size': 5000000, 'language': 'en'}
multilingual  -> {'size': 5000000, 'language': ['en', 'es', 'fr']}

registry di disk (variants.json):
{
  "base": {
    "base": null,
    "changes": {
      "size": 1000000,
      "language": "en"
    }
  },
  "large": {
    "base": "base",
    "changes": {
      "size": 5000000
    }
  },
  "multilingual": {
    "base": "large",
    "changes": {
      "language": [
        "en",
        "es",
        "fr"
      ]
    }
  }
}


`multilingual` mewarisi `size` dari `large` (yang sudah menimpa milik `base`) dan menimpa `language` dengan miliknya sendiri; aturannya sama dengan merge dict Python, nilai paling kanan yang menang. Registry di disk tetap hanya menyimpan perbedaan, dan `assert` terakhir memastikan registry itu terbaca ulang utuh oleh manager baru. Dua batas yang perlu diingat: penimpaan bekerja per key utuh (daftar `language` mengganti, bukan menambah, nilai induk), dan registry yang diedit tangan bisa membentuk siklus. Parameter `_chain` membuat siklus berhenti dengan pesan yang jelas, bukan `RecursionError`.

Untuk penamaan, buku menyarankan pola `<base>_<modifier1>_<modifier2>_..._<description>`, misalnya `clean_5m_multilingual_ablation2`. Nama memudahkan manusia membaca, registry memberi resolusi yang pasti untuk mesin.

## 7. Harga membaca rantai delta · CPU/I/O

Delta menumpuk: versi ke-N direkonstruksi dengan menerapkan N patch berturut-turut ke base. Penyimpanan jadi murah, tetapi membaca versi yang jauh makin mahal. Eksperimen ini membuat 50 revisi dari 4.000 baris, setiap revisi mengubah 20 baris acak. Base dan setiap patch (format opcode dari bagian 2) disimpan sebagai file, begitu juga snapshot versi ke-50. Lalu kita ukur waktu membaca versi ke-N lewat rantai, dibandingkan dengan membaca satu snapshot langsung dari disk.

`ndiff` sengaja tidak dipakai di sini. Setiap diff `ndiff` sudah memuat seluruh versi barunya, jadi ia tidak pernah membentuk rantai yang sesungguhnya; cukup diff terakhir yang dibaca.

In [27]:
CHAIN_DIR = WORK_DIR / "delta_chain"
shutil.rmtree(CHAIN_DIR, ignore_errors=True)
CHAIN_DIR.mkdir()
rng = np.random.default_rng(7)
versions = [[f'{{"id": {i}, "text": "dokumen {i} ' + "lorem ipsum " * 8 + '"}' for i in range(4000)]]
for step in range(50):
    curr = list(versions[-1])
    for j in rng.choice(len(curr), size=20, replace=False):
        curr[j] = curr[j][:-2] + f' rev{step}"}}'
    (CHAIN_DIR / f"patch_{step + 1:02d}.json").write_text(json.dumps(make_patch(versions[-1], curr)))
    versions.append(curr)
(CHAIN_DIR / "base.txt").write_text("\n".join(versions[0]))
(CHAIN_DIR / "snapshot_v50.txt").write_text("\n".join(versions[50]))


def read_via_chain(n):
    state = (CHAIN_DIR / "base.txt").read_text().split("\n")
    for k in range(1, n + 1):
        state = apply_patch(state, json.loads((CHAIN_DIR / f"patch_{k:02d}.json").read_text()))
    return state


def read_snapshot():
    return (CHAIN_DIR / "snapshot_v50.txt").read_text().split("\n")


def best_of_3(fn):
    times = []
    for _ in range(3):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return min(times)


assert read_via_chain(50) == read_snapshot() == versions[50]
t_snapshot = best_of_3(read_snapshot)
print(f"{'versi ke-N':>10} | {'rantai (ms)':>11} | {'x snapshot':>10}")
for n in (1, 5, 10, 25, 50):
    t_chain = best_of_3(lambda: read_via_chain(n))
    print(f"{n:>10} | {t_chain * 1000:>11.2f} | {t_chain / t_snapshot:>10.1f}")
print(f"{'snapshot':>10} | {t_snapshot * 1000:>11.2f} |")

all_snapshots = sum(len("\n".join(v).encode()) for v in versions[1:])
chain_bytes = sum(p.stat().st_size for p in CHAIN_DIR.glob("patch_*.json")) + (CHAIN_DIR / "base.txt").stat().st_size
print(f"\nstorage 50 versi: 50 snapshot = {all_snapshots / 2**20:.1f} MB | base + 50 patch = {chain_bytes / 2**20:.2f} MB")

versi ke-N | rantai (ms) | x snapshot
         1 |        0.85 |        1.2
         5 |        1.28 |        1.8
        10 |        1.75 |        2.4
        25 |        3.39 |        4.6
        50 |        5.89 |        8.1
  snapshot |        0.73 |

storage 50 versi: 50 snapshot = 25.6 MB | base + 50 patch = 0.67 MB


`assert` di awal memastikan rantai 50 patch menghasilkan versi yang persis sama dengan snapshot-nya. Kolom `x snapshot` menunjukkan berapa kali lebih lama membaca versi ke-N lewat rantai dibanding membaca snapshot. Setiap `apply_patch` menyalin seluruh 4.000 baris dan setiap patch adalah satu file yang harus dibuka, jadi biayanya tumbuh kira-kira linear terhadap N, bukan terhadap jumlah baris yang berubah. Di sisi storage polanya terbalik: base ditambah 50 patch jauh lebih kecil daripada 50 snapshot.

Dari dua arah inilah resep sistem yang matang: delta ringkas ditambah snapshot berkala. Git membatasi kedalaman rantai delta di packfile, dan database menyebut snapshot berkalanya *checkpoint*. DVC memilih ekstrem yang lain (bagian 3): setiap versi file adalah objek utuh, sehingga membaca selalu murah dan storage yang dibayar. Tidak ada jawaban tunggal; pilihannya bergantung pada seberapa sering dataset-mu ditulis dibanding dibaca.

## Ringkasan

- **Identitas.** Hash SHA-256 atas JSON dengan key terurut memberi ID yang sama untuk isi yang sama, dan verifikasi saat `load` menangkap file yang diedit tangan.
- **Delta.** `ndiff` + `restore` bisa dipulihkan tetapi tidak pernah lebih kecil dari snapshot; patch dari opcode `SequenceMatcher` ringkas sekaligus bisa diterapkan. Rantai delta hemat storage tetapi makin mahal dibaca, jadi butuh snapshot berkala.
- **DVC.** Pointer kecil, cache yang dialamatkan hash, dan remote. Rollback berarti mengembalikan pointer lalu `dvc checkout`, dibuktikan dengan MD5. Cache menyimpan versi utuh per file, jadi satu baris yang berubah tetap dibayar satu file penuh.
- **Training.** Hash versi data masuk ke checkpoint model yang benar-benar dilatih, dan data yang tidak cocok dengan versi tercatat memblokir training.
- **Korpus dan varian.** Manifest SHA-256 mendeteksi file yang ditambah, dihapus, dan diubah; registry varian menyimpan perbedaan konfigurasi dan menggabungkannya secara rekursif.

Praktik terbaik dari buku yang kini punya contoh berjalan di notebook ini: pakai perkakas khusus seperti DVC, tanam versi data di metadata model, cadangkan data ke remote dengan `dvc push`, pakai konvensi nama varian yang konsisten, dan periksa versi data di CI/CD sebelum training dimulai.

**Bab berikutnya:** anotasi dan pelabelan dataset untuk LLM.